# U.S. Commuter Rail Benchmarking — FY2023
**Project 1: U.S. Rail Capacity Benchmarking Dashboard**

This notebook walks through the full analytical workflow:
1. Load and inspect raw NTD + agency data
2. Compute 6 KPIs and a weighted composite score
3. Generate all figures for the README and technical brief
4. Key findings and infrastructure recommendations

**Systems analyzed:** MBTA, NJ Transit, SEPTA, Metra, Caltrain, MARC  
**Data source:** NTD FY2023 Annual Data + agency OTP dashboards

In [20]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Import our analysis module
from analysis.benchmark_analysis import load_data, compute_kpis, AGENCY_COLORS, PALETTE, WEIGHTS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Environment ready.')

Environment ready.


## 1. Load Data

In [21]:
df_raw = load_data('../data/raw_data.csv')
print(f'{len(df_raw)} agencies loaded')
df_raw.head(10)

6 agencies loaded


,agency,ntd_id,region,upt,vrm,vrh,voms,vams,route_miles,opex_usd,otp,peak_trains_per_dir,avg_train_seats
0,MBTA,10001,Northeast,32500000,19800000,890000,392,450,415.0,512000000,0.887,22,900
1,NJ Transit,20215,Northeast,75200000,43100000,1650000,744,830,1010.0,1320000000,0.921,34,1100
2,SEPTA,30030,Mid-Atlantic,35100000,21300000,920000,408,470,280.0,598000000,0.896,18,850
3,Metra,50077,Midwest,28400000,32700000,1180000,510,590,490.0,748000000,0.918,15,950
4,Caltrain,90026,West Coast,9800000,8100000,280000,90,105,77.0,198000000,0.912,10,700
5,MARC,30034,Mid-Atlantic,8900000,11200000,390000,150,175,187.0,234000000,0.883,12,800


## 2. Compute KPIs
Six derived metrics + weighted composite score (0–100).

| KPI | Formula | Weight | Direction |
|-----|---------|--------|-----------|
| Ridership Density | UPT / Route Miles | 25% | Higher = better |
| Fleet Utilization | VOMS / VAMS | 15% | Higher = better |
| Cost per Trip | OpEx / UPT | 20% | **Lower = better** |
| Service Intensity | VRM / Route Miles | 15% | Higher = better |
| Rev. Hr Efficiency | UPT / VRH | 15% | Higher = better |
| On-Time Performance | Agency-reported | 10% | Higher = better |

In [22]:
df = compute_kpis(df_raw)

# Show computed KPIs
display_cols = ['agency', 'ridership_density', 'fleet_utilization',
                'cost_per_trip', 'service_intensity', 'rev_hr_efficiency',
                'otp', 'composite_score', 'overall_rank', 'tier']
df[display_cols].style \
    .format({'ridership_density': '{:,.0f}', 'fleet_utilization': '{:.1%}',
             'cost_per_trip': '${:.2f}', 'service_intensity': '{:,.1f}',
             'rev_hr_efficiency': '{:,.1f}', 'otp': '{:.1%}',
             'composite_score': '{:.1f}'}) \
    .background_gradient(subset=['composite_score'], cmap='RdYlGn') \
    .background_gradient(subset=['cost_per_trip'], cmap='RdYlGn_r')

,agency,ridership_density,fleet_utilization,cost_per_trip,service_intensity,rev_hr_efficiency,otp,composite_score,overall_rank,tier
0,SEPTA,"125,357",86.8%,$17.04,"76,071.4",38.2,89.6%,67.7,1,Average
1,Caltrain,"127,273",85.7%,$20.20,"105,194.8",35.0,91.2%,67.3,2,Average
2,NJ Transit,"74,455",89.6%,$17.55,"42,673.3",45.6,92.1%,65.0,3,Average
3,MBTA,"78,313",87.1%,$15.75,"47,710.8",36.5,88.7%,46.3,4,Below Average
4,Metra,"57,959",86.4%,$26.34,"66,734.7",24.1,91.8%,21.8,5,Needs Improvement
5,MARC,"47,594",85.7%,$26.29,"59,893.0",22.8,88.3%,4.2,6,Needs Improvement


## 3. Key Findings
Run this cell after entering real data — findings update automatically.

In [23]:
top = df.iloc[0]
bottom = df.iloc[-1]
highest_density = df.loc[df['ridership_density'].idxmax()]
lowest_cost = df.loc[df['cost_per_trip'].idxmin()]
best_otp = df.loc[df['otp'].idxmax()]

print('=== KEY FINDINGS ===')
print(f'Top overall performer:    {top["agency"]} (score: {top["composite_score"]:.1f})')
print(f'Bottom performer:         {bottom["agency"]} (score: {bottom["composite_score"]:.1f})')
print(f'Highest ridership density:{highest_density["agency"]} ({highest_density["ridership_density"]:,.0f} trips/route mile)')
print(f'Lowest cost per trip:     {lowest_cost["agency"]} (${lowest_cost["cost_per_trip"]:.2f}/trip)')
print(f'Best on-time performance: {best_otp["agency"]} ({best_otp["otp"]:.1%})')

# Cost range
cost_range = df['cost_per_trip'].max() - df['cost_per_trip'].min()
print(f'\nCost/trip spread across systems: ${cost_range:.2f}')
print(f'OTP range: {df["otp"].min():.1%} – {df["otp"].max():.1%}')

=== KEY FINDINGS ===
Top overall performer:    SEPTA (score: 67.7)
Bottom performer:         MARC (score: 4.2)
Highest ridership density:Caltrain (127,273 trips/route mile)
Lowest cost per trip:     MBTA ($15.75/trip)
Best on-time performance: NJ Transit (92.1%)

Cost/trip spread across systems: $10.58
OTP range: 88.3% – 92.1%


## 4. Visualizations

In [24]:
# Run all figures inline
from analysis.benchmark_analysis import (
    fig_composite_score, fig_kpi_rankings, fig_correlation,
    fig_cost_vs_otp, fig_ridership_density, fig_radar
)
OUT = '../outputs'
os.makedirs(OUT, exist_ok=True)

fig_composite_score(df, OUT)
fig_kpi_rankings(df, OUT)
fig_correlation(df, OUT)
fig_cost_vs_otp(df, OUT)
fig_ridership_density(df, OUT)
fig_radar(df, OUT)
print('All figures saved to /outputs')

  ✓  fig1_composite_score.png
  ✓  fig2_kpi_rankings_heatmap.png
  ✓  fig3_correlation_heatmap.png
  ✓  fig4_cost_vs_otp.png
  ✓  fig5_ridership_density.png
  ✓  fig6_radar_chart.png
All figures saved to /outputs


## 5. Statistical Analysis — What Drives Composite Score?

In [25]:
# Pearson correlation of each KPI with composite score
kpis = list(WEIGHTS.keys())
print('Correlation with Composite Score:')
print('-' * 40)
for kpi in kpis:
    r, p = stats.pearsonr(df[kpi], df['composite_score'])
    sig = '**' if p < 0.05 else ('*' if p < 0.10 else '')
    print(f'  {kpi:<25} r = {r:+.3f}  p = {p:.3f} {sig}')
print('\n** p<0.05  * p<0.10')

Correlation with Composite Score:
----------------------------------------
  ridership_density         r = +0.835  p = 0.039 **
  fleet_utilization         r = +0.437  p = 0.386 
  cost_per_trip             r = -0.824  p = 0.044 **
  service_intensity         r = +0.278  p = 0.594 
  rev_hr_efficiency         r = +0.884  p = 0.019 **
  otp                       r = +0.387  p = 0.449 

** p<0.05  * p<0.10


## 6. Infrastructure Recommendations
> Based on the benchmarking analysis — to be included in the technical brief

In [26]:
# Auto-generate recommendation bullets based on data
high_cost = df[df['cost_per_trip'] > df['cost_per_trip'].median()]['agency'].tolist()
low_otp   = df[df['otp'] < df['otp'].median()]['agency'].tolist()
low_util  = df[df['fleet_utilization'] < 0.80]['agency'].tolist()

print('Data-driven recommendations:')
if high_cost:
    print(f'\n1. COST REDUCTION ({", ".join(high_cost)})')
    print('   Operating costs above system median — review maintenance contracts,')
    print('   crew scheduling efficiency, and purchased transportation agreements.')
if low_otp:
    print(f'\n2. RELIABILITY INVESTMENT ({", ".join(low_otp)})')
    print('   OTP below system median — priority candidates for signal')
    print('   infrastructure upgrades, track renewal, and schedule padding review.')
if low_util:
    print(f'\n3. FLEET RIGHT-SIZING ({", ".join(low_util)})')
    print('   Fleet utilization below 80% — excess reserve vehicles; assess')
    print('   whether VAMS can be reduced through shared maintenance agreements.')

Data-driven recommendations:

1. COST REDUCTION (Caltrain, Metra, MARC)
   Operating costs above system median — review maintenance contracts,
   crew scheduling efficiency, and purchased transportation agreements.

2. RELIABILITY INVESTMENT (SEPTA, MBTA, MARC)
   OTP below system median — priority candidates for signal
   infrastructure upgrades, track renewal, and schedule padding review.
